<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Cartopy_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
             opt_url = 'https://earthengine-highvolume.googleapis.com'
              )

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
pdsi = (ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
    .filterDate("2010", "2025")
    .select("pdsi")
    .mean()
)
pdsi

In [ ]:
import xarray as xr

In [ ]:
ds = xr.open_dataset(
    pdsi,
    engine = "ee",
    crs = "EPSG:4326",
    geometry = roi,
    scale = 1

)
ds

In [ ]:
ds = ds.squeeze("time").drop_vars("time") * 0.01

In [ ]:
!pip install cartopy

In [ ]:
import cartopy.crs as ccrs

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
ds

In [ ]:
plt.figure()
ax = plt.axes(projection = ccrs.EqualEarth())
ds.pdsi.plot(
    x = "lon", y = "lat", transform =ccrs.PlateCarree(),
    robust = True, cmap = "RdBu", levels = 15, cbar_kwargs = {
        "orientation": "horizontal", "pad": 0.08, "shrink":0.7
    }
)

ax.coastlines(resolution = "110m")
ax.set_global()
ax.gridlines(draw_labels = True, alpha = 0.2, color = 'gray')
ax.set_title("Global Drought Map (2010-2025)", fontsize = 15)

plt.savefig("drought_map.png", dpi = 360, bbox_inches = "tight")

In [ ]:
# To plot global artisanal gold mining locations, you would first need a dataset
# of these locations, which could be points or polygons. A comprehensive,
# globally consistent dataset for artisanal gold mining isn't readily available
# as a standard Earth Engine image collection like 'TERRACLIMATE'.

# However, if you have specific coordinates or a dataset (e.g., an ee.FeatureCollection
# or a GeoPandas GeoDataFrame), you can overlay it on your Cartopy map.

# For demonstration, let's add some hypothetical artisanal gold mining locations
# as scatter points on a new map that also displays the PDSI data.

!pip install cartopy # Ensure cartopy is installed

# Re-import necessary libraries for clarity in this new cell
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np # For generating hypothetical points

plt.figure(figsize=(15, 10)) # Adjust figure size for better visibility
ax = plt.axes(projection=ccrs.EqualEarth())

# Plot the existing PDSI data
# Make sure 'ds' (your xarray Dataset) is accessible if you run this independently
# or if you are recreating the map after 'ds' has been defined.
if 'ds' in locals() or 'ds' in globals():
    ds.pdsi.plot(
        x="lon", y="lat", transform=ccrs.PlateCarree(),
        robust=True, cmap="RdBu", levels=15, cbar_kwargs={
            "orientation": "horizontal", "pad": 0.08, "shrink":0.7
        },
        ax=ax # Crucial: plot on the current axes
    )
else:
    print("Warning: 'ds' (PDSI data) not found. Plotting only base map and mining sites.")

# Add base map features
ax.coastlines(resolution="110m", linewidth=0.8, color='black') # Thicker coastlines
ax.set_global()
ax.gridlines(draw_labels=True, alpha=0.3, color='grey', linestyle='--') # More subtle gridlines
ax.set_title("Global Drought Map with Hypothetical Artisanal Gold Mining Sites (2010-2025)", fontsize=16)

# --- Add hypothetical artisanal gold mining locations in specified regions ---
np.random.seed(42) # for reproducibility
num_total_mining_sites = 3000 # Increased to ensure good coverage across all specified regions
all_mining_lons = np.random.uniform(-180, 180, num_total_mining_sites)
all_mining_lats = np.random.uniform(-70, 70, num_total_mining_sites)

# Define bounding boxes for specified regions (min_lon, max_lon, min_lat, max_lat)
# These are approximate bounding boxes for the requested regions
regions = {
    "Brazil": (-75, -34, -34, 6),
    "West Africa": (-18, 15, 4, 20),
    "Central Africa": (5, 30, -10, 10),
    "Southern Africa": (10, 40, -35, -10),
    "Southeast Asia": (95, 140, -10, 20),
    "China": (75, 135, 18, 50),
    "Europe": (-10, 40, 35, 70)
}

filtered_lons_serious = []
filtered_lats_serious = []
filtered_lons_not_serious = []
filtered_lats_not_serious = []

for region_name, (min_lon, max_lon, min_lat, max_lat) in regions.items():
    # Create a mask for points within the current region
    mask_lon = (all_mining_lons >= min_lon) & (all_mining_lons <= max_lon)
    mask_lat = (all_mining_lats >= min_lat) & (all_mining_lats <= max_lat)
    region_mask = mask_lon & mask_lat

    # Get the points within the current region
    region_lons = all_mining_lons[region_mask]
    region_lats = all_mining_lats[region_mask]

    # Randomly assign 'seriousness' to these points
    # Let's say 30% are 'serious' (red), 70% are 'not serious' (blue)
    is_serious = np.random.rand(len(region_lons)) < 0.3

    filtered_lons_serious.extend(region_lons[is_serious])
    filtered_lats_serious.extend(region_lats[is_serious])
    filtered_lons_not_serious.extend(region_lons[~is_serious])
    filtered_lats_not_serious.extend(region_lats[~is_serious])

# Plot 'serious' mining sites in red
ax.scatter(
    filtered_lons_serious, filtered_lats_serious,
    transform=ccrs.PlateCarree(), # Ensure coordinates are interpreted correctly
    color='red',
    s=70, # Size of the markers
    marker='^', # Use a triangle marker for mines
    alpha=0.8,
    edgecolors='darkred', # Outline color
    linewidths=1.0,
    label='Serious Artisanal Gold Mining Sites (Hypothetical)'
)

# Plot 'not serious' mining sites in blue
ax.scatter(
    filtered_lons_not_serious, filtered_lats_not_serious,
    transform=ccrs.PlateCarree(), # Ensure coordinates are interpreted correctly
    color='blue',
    s=70, # Size of the markers
    marker='^', # Use a triangle marker for mines
    alpha=0.8,
    edgecolors='darkblue', # Outline color
    linewidths=1.0,
    label='Not Serious Artisanal Gold Mining Sites (Hypothetical)'
)

plt.legend(loc='lower left', frameon=True, fancybox=True, shadow=True, borderpad=1) # Add a legend
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.savefig("drought_and_hypothetical_mining_map_categorized.png", dpi=360, bbox_inches="tight")
plt.show()